In [ ]:
#Setting up the Satscan environment.
from google.colab import files
uploaded = files.upload()

In [ ]:
#Setting up the Satscan environment.
!gunzip satscan.10.3.3.linux.tar.gz
!tar -xvf satscan.10.3.3.linux.tar
!chmod +x satscan64

In [ ]:
#Setting up directories for inputs and outputs.
import os

os.makedirs("satscan_inputs", exist_ok=True)
os.makedirs("satscan_outputs", exist_ok=True)

In [ ]:
#Load the zip files with datasets.
from google.colab import files
uploaded = files.upload()

In [ ]:
#Unzip the datasets.
!mkdir -p datasets27new4
!unzip KidneyData.zip -d datasets27new4/

In [ ]:
#Load the coordinate file (NMgeo.csv).
from google.colab import files
uploaded = files.upload()

In [ ]:
#Save coordinate file as coords.
import pandas as pd
coords = pd.read_csv("NMgeo.csv")

In [ ]:
#Load the parameter file (prm4.txt)
from google.colab import files
uploaded = files.upload()

In [ ]:
import os
import pandas as pd

def create_inputoutputfiles(df):
  # Merge with coordinates
  df = df.merge(coords, left_on="region", right_on="county", how="left")
  # File paths
  case_file  = os.path.abspath(f"{"satscan_inputs"}/case.txt")
  pop_file   = os.path.abspath(f"{"satscan_inputs"}/pop.txt")
  coord_file = os.path.abspath(f"{"satscan_inputs"}/coord.txt")
  time_file  = os.path.abspath(f"{"satscan_inputs"}/time.txt")
  #prm_file   = os.path.abspath(f"{"satscan_inputs"}/params_1.prm")
  results_file = os.path.join("satscan_outputs", f"results.txt")

  # Write input files (include 'year' in case and population files)
  df[['States','Count','Year']].to_csv(case_file, sep=" ", index=False, header=False)
  df[['States','Year','Population']].to_csv(pop_file, sep=" ", index=False, header=False)
  df[['States','x-coordinate','y-coordinate']].to_csv(coord_file, sep=" ", index=False, header=False)
  df[['States','Year']].to_csv(time_file, sep=" ", index=False, header=False)

dataset_files = sorted(os.listdir(""))

# Filter out directories and non-csv files
csv_files = [f for f in dataset_files if os.path.isfile(os.path.join("", f)) and f.endswith('.csv')]

#Create input and output files for datasets
for i, file in enumerate(csv_files):
    df = pd.read_csv(f"/{file}")
    create_inputoutputfiles(df, i)

In [ ]:
#Function to update the prm file and run
def updateAndrunprm(index):
  prm_in = "/content/prm4.txt"
  prm_out = "/content/updatedprm1.prm"

  case_file     = os.path.abspath(f"/content/satscan_inputs/case_{index}.txt")
  pop_file      = os.path.abspath(f"/content/satscan_inputs/pop_{index}.txt")
  coord_file    = os.path.abspath(f"/content/satscan_inputs/coord_{index}.txt")
  time_file     = os.path.abspath(f"/content/satscan_inputs/time_{index}.txt")
  results_file  = os.path.abspath(f"/content/satscan_inputs/result_{index}.txt")

  # Read prm file
  with open(prm_in, "r") as f:
      prm = f.read()

  def replace_or_add(text, key, value):
      pattern = rf"^{key}\s*=.*$"
      replacement = f"{key}={value}"
      if re.search(pattern, text, flags=re.MULTILINE):
          return re.sub(pattern, replacement, text, flags=re.MULTILINE)
      else:
          return text + f"\n{replacement}"

  # Update parameters
  prm = replace_or_add(prm, "CaseFile", case_file)
  prm = replace_or_add(prm, "PopulationFile", pop_file)
  prm = replace_or_add(prm, "CoordinatesFile", coord_file)
  prm = replace_or_add(prm, "TimeFile", time_file)
  prm = replace_or_add(prm, "ResultsFile", results_file)

  # Write updated prm
  with open(prm_out, "w") as f:
      f.write(prm)

  !./satscan64 /content/updatedprm1.prm


In [ ]:
#Run the method for each dataset
import re
for i,file in enumerate(csv_files):
  updateAndrunprm(i)

In [ ]:
#Define the true cluster

import pandas as pd

true_cluster = [""]
print("True cluster region years:")
print(true_cluster)

In [ ]:
#Load the population file
from google.colab import files
uploaded = files.upload()

In [ ]:
#Read the population file
populationDF = pd.read_csv("datapop.csv")

In [ ]:
import re

def expand_significant_clusters(results_file, p_threshold=0.1):
    with open(results_file, 'r', encoding='utf-8') as f:
        content = f.read()

    # Split by cluster numbers (e.g., "1.", "2.", "3.") followed by "Location IDs"
    # This is a safer way to isolate each cluster block
    cluster_blocks = re.split(r'\n\s*(\d+)\.Location IDs included', content)

    combined_region_years = []

    # Iterate through blocks (skipping index 0 which is the header)
    # Since re.split with a group returns the digit too, we step by 2
    for i in range(2, len(cluster_blocks), 2):
        block = cluster_blocks[i]

        # 1. Extract P-value (handles any amount of dots or spaces)
        p_match = re.search(r'P-value[\.\s]*:\s*([\d\.]+)', block)
        if not p_match:
            continue

        p_value = float(p_match.group(1))

        if p_value < p_threshold:
            # 2. Extract Location IDs (Everything before the Coordinates line)
            # We look at the very beginning of the block content
            loc_section = block.split("Coordinates")[0]
            # Remove the initial ".: " and then split by commas
            loc_string = loc_section.lstrip(".: \t")
            regions = [x.strip() for x in loc_string.split(",")]

            # 3. Extract Time Frame (handles any amount of dots)
            time_match = re.search(r'Time frame[\.\s]*:\s*(\d{4})/\d+/\d+ to (\d{4})/\d+/\d+', block)
            if time_match:
                start_year = int(time_match.group(1))
                end_year = int(time_match.group(2))

                for region in regions:
                    for year in range(start_year, end_year + 1):
                        combined_region_years.append(f"{region}_{year}")

    return list(set(combined_region_years))

def calc_sensitivity_ppv_region_year(true_cluster_years, detected_region_years):
      """
      Compute sensitivity and PPV for expanded region-year clusters.
      """

      true_set = set(true_cluster_years)
      detected_set = set(detected_region_years)

      pop_sumTRUE = populationDF[populationDF['pair_id'].isin(true_set)]['total_population'].sum()
      pop_sumDETECTED = populationDF[populationDF['pair_id'].isin(detected_set)]['total_population'].sum()
      pop_sumMUTUAL = populationDF[populationDF['pair_id'].isin(true_set&detected_set)]['total_population'].sum()

      sensitivity = pop_sumMUTUAL / pop_sumTRUE if pop_sumTRUE > 0 else None
      ppv = pop_sumMUTUAL / pop_sumDETECTED if pop_sumDETECTED > 0 else None

      return sensitivity, ppv


def evaluate_satscan_region_year(results_file, true_cluster_years):
    """
    Wrapper: Extract top 5 clusters, expand by year, and compute sensitivity & PPV.
    """
    detected_region_years = expand_significant_clusters(results_file)
    sensitivity, ppv = calc_sensitivity_ppv_region_year(true_cluster_years, detected_region_years)

    return {
        "detected_region_years": detected_region_years,
        "sensitivity": sensitivity,
        "PPV": ppv
    }


In [ ]:
import re
import pandas as pd
rows = []
for i,file in enumerate(dataset_files):
  results_file = f"/content/satscan_inputs/result_{i}.txt"
  output = evaluate_satscan_region_year(results_file, true_cluster)
  # Extract sensitivity and PPV
  sen = output["sensitivity"]
  ppv = output["PPV"]
  # Append a row: [i, sensitivity, PPV]
  rows.append([i, sen, ppv])
  # Convert all rows into a DataFrame
  results_df = pd.DataFrame(rows, columns=["dataset_index", "sensitivity", "PPV"])

results_df.to_csv("", index=False)

In [ ]:

# Print average results
avg_sensitivityCylindricalMethod = results_df["sensitivity"].mean()
avg_ppvCylindricalMethod = results_df["PPV"].mean()


print("Average Sensitivity Cylindrical method:", avg_sensitivityCylindricalMethod)
print("Average PPV Cylindrical method:", avg_ppvCylindricalMethod)